# 04. FP8 (E4M3, E5M2, FNUZ) and Minifloats (FP6, FP4)

- **Category:** 13_precision_and_numerics
- **Lineage & Citations:** Micikevicius et al. (2022, *FP8 Formats for Deep Learning*, NVIDIA/ARM/Intel); OCP (2023, *Microscaling Formats (MX) Specification*); Dettmers et al. (2023, *QMoE: Practical Sub-1-Bit Compression of Trillion-Parameter Models*).
- **Purpose:** Formulate the bit layouts, dynamic range trade-offs, and delayed-scaling GEMM mechanics of modern sub-8-bit floating point formats (FP8 E4M3FN, FP8 E5M2, FP6, and FP4 E2M1).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [03. Automatic Mixed Precision Autocast and GradScaler](03_automatic_mixed_precision_autocast_and_gradscaler.ipynb) $\to$ `🔗 Downstream Syntheses:` [05. Block Floating Point BFP Microscaling MXFP and NVFP4](05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT use E5M2 for forward activations or static weights (its 2 mantissa bits degrade Signal-to-Quantization-Noise Ratio below acceptable loss thresholds); do NOT use E4M3 for unscaled gradients (its narrow dynamic range of $[-448, 448]$ underflows deep network gradients to zero).

## Mathematical Foundations & Minifloat Formulations

1. **Minifloat Quantization Formula**:
   $$x = (-1)^s \times 2^{e - \text{bias}} \times (1 + m \cdot 2^{-M}), \quad \Delta = \begin{cases} 2^{1 - \text{bias} - M}, & e < 1 - \text{bias} \text{ (subnormal)} \\ 2^{e - M}, & e \ge 1 - \text{bias} \text{ (normal)} \end{cases}$$
2. **FP8 E4M3FN Configuration**: $E=4, M=3, \text{bias}=7, \text{max}=448.0$.
3. **FP8 E5M2 Configuration**: $E=5, M=2, \text{bias}=15, \text{max}=57344.0$.
4. **FP4 E2M1 Configuration**: $E=2, M=1, \text{bias}=1, \text{max}=6.0 \implies \{0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0\}$.
5. **Delayed Scaling Tensor Core GEMM**:
   $$\mathbf{Y} = (\tilde{\mathbf{A}} \cdot \tilde{\mathbf{B}}) \times (s_A \cdot s_B), \quad s_{A, t+1} = \frac{\max(|\mathbf{A}_t|)}{\text{max\_val}}$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/13_precision_and_numerics/04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def linear_anchor(x, weight, bias):
    return x @ weight + bias

def quantize_minifloat(x, e_bits, m_bits, bias, max_val):
    x_abs = torch.abs(x)
    is_zero = (x_abs == 0.0)
    x_safe = torch.where(is_zero, torch.ones_like(x_abs), x_abs)
    e = torch.floor(torch.log2(x_safe))
    e_min_normal = 1.0 - float(bias)
    step = torch.where(
        e < e_min_normal,
        2.0 ** (e_min_normal - float(m_bits)),
        2.0 ** (e - float(m_bits)),
    )
    q = torch.round(x_safe / step) * step
    q = torch.clamp(q, min=0.0, max=float(max_val))
    return torch.where(is_zero, torch.zeros_like(x), torch.sign(x) * q)

def fp8_e4m3fn_quantize(x, scale=1.0):
    return quantize_minifloat(x / float(scale), 4, 3, 7, 448.0) * float(scale)

def fp8_e5m2_quantize(x, scale=1.0):
    return quantize_minifloat(x / float(scale), 5, 2, 15, 57344.0) * float(scale)

def fp4_e2m1_quantize(x, scale=1.0):
    return quantize_minifloat(x / float(scale), 2, 1, 1, 6.0) * float(scale)

def delayed_scaled_fp8_gemm(a, b, a_scale, b_scale):
    a_q = fp8_e4m3fn_quantize(a, scale=a_scale)
    b_q = fp8_e4m3fn_quantize(b, scale=b_scale)
    out = a_q @ b_q
    next_a = max(1e-6, float(torch.max(torch.abs(a))) / 448.0)
    next_b = max(1e-6, float(torch.max(torch.abs(b))) / 448.0)
    return out, next_a, next_b

In [ ]:
# 1. Linear anchor parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert torch.max(torch.abs(y_anchor - weights["test_output"])) < 1e-6

# 2. FP8 E4M3FN Quantization Parity
e4m3 = fp8_e4m3fn_quantize(weights["fp8_input"], scale=0.5)
assert torch.max(torch.abs(e4m3 - weights["e4m3_ref"])) < 1e-5

# 3. FP8 E5M2 Quantization Parity
e5m2 = fp8_e5m2_quantize(weights["fp8_input"], scale=0.5)
assert torch.max(torch.abs(e5m2 - weights["e5m2_ref"])) < 1e-5

# 4. FP4 E2M1 Quantization Parity
fp4 = fp4_e2m1_quantize(weights["fp4_input"], scale=0.25)
assert torch.max(torch.abs(fp4 - weights["fp4_ref"])) < 1e-5

# 5. Delayed Scaled FP8 GEMM Parity
gemm_out, n_a, n_b = delayed_scaled_fp8_gemm(weights["gemm_a"], weights["gemm_b"], a_scale=0.01, b_scale=0.01)
assert torch.max(torch.abs(gemm_out - weights["gemm_out_ref"])) < 1e-5
assert abs(n_a - float(weights["next_scale_a"])) < 1e-5
assert abs(n_b - float(weights["next_scale_b"])) < 1e-5
print("All FP8 (E4M3, E5M2), FP4, and Delayed GEMM parity checks passed!")

In [ ]:
# Microbenchmark: E4M3 vs FP4 simulation throughput
x_bench = weights["fp8_input"]
t0 = time.perf_counter()
for _ in range(500):
    _ = fp8_e4m3fn_quantize(x_bench, scale=0.5)
e4m3_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"FP8 E4M3 quantization latency: {e4m3_us:.2f} us")

t0 = time.perf_counter()
for _ in range(500):
    _ = fp4_e2m1_quantize(weights["fp4_input"], scale=0.25)
fp4_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"FP4 E2M1 quantization latency: {fp4_us:.2f} us")

In [ ]:
# Pathological Misuse: E5M2 vs E4M3 Reconstruction Error for Gaussian Weights
torch.manual_seed(123)
weights_normal = torch.randn(1000)

# E4M3FN (3 mantissa bits) vs E5M2 (2 mantissa bits)
q_e4m3 = fp8_e4m3fn_quantize(weights_normal, scale=1.0)
q_e5m2 = fp8_e5m2_quantize(weights_normal, scale=1.0)

rmse_e4m3 = float(torch.sqrt(torch.mean((weights_normal - q_e4m3) ** 2)))
rmse_e5m2 = float(torch.sqrt(torch.mean((weights_normal - q_e5m2) ** 2)))

print(f"E4M3 Reconstruction RMSE: {rmse_e4m3:.4f}")
print(f"E5M2 Reconstruction RMSE: {rmse_e5m2:.4f}")
assert rmse_e4m3 < rmse_e5m2, "E4M3 must yield strictly lower reconstruction error on weights than E5M2!"
print(f"Precision advantage of E4M3: {(rmse_e5m2 / rmse_e4m3 - 1.0) * 100:.1f}% lower RMSE")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Format Specialization**: E4M3FN allocates 3 mantissa bits for weights and activations where precision is paramount; E5M2 allocates 5 exponent bits for backpropagated gradients where dynamic range is vital.
2. **Delayed Scaling**: Asynchronous tracking of maximum absolute value $\operatorname{amax}$ removes GPU synchronization overhead from the critical GEMM execution path.
3. **Minifloat Hierarchy**: FP6 and FP4 extend floating point dynamics into extreme compression regimes, maintaining non-linear representation superior to uniform integer grids.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.md](../../markdown/13_precision_and_numerics/04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.md)
- **PyTorch 2.x Module:** [04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.py](../../pytorch2/13_precision_and_numerics/04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.py)
- **Functional JAX Module:** [04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.py](../../jax/13_precision_and_numerics/04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.py)
- **Native Rust Candle Track:** [04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.rs](../../candle/13_precision_and_numerics/04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.rs)
- **Hardware Kernel (CPU / WGSL):** [04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.rs](../../kernels/13_precision_and_numerics/04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.rs)
- **Golden SafeTensors:** [04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.safetensors](../../weights/13_precision_and_numerics/04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 03. Automatic Mixed Precision Autocast and GradScaler](03_automatic_mixed_precision_autocast_and_gradscaler.ipynb) | [Master Index](../../README.md#13_precision_and_numerics) | [Next: 05. Block Floating Point BFP Microscaling MXFP and NVFP4](05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.ipynb) &rarr;